In [0]:
from pyspark.sql import functions as F

In [0]:
%sql
--silver
DROP TABLE IF EXISTS workspace.silver.tbl_representantes;
DROP TABLE IF EXISTS workspace.silver.tbl_productos;
DROP TABLE IF EXISTS workspace.silver.tbl_ventas_detalle;

--quarantine
DROP TABLE IF EXISTS workspace.quarantine.tbl_representantes_rechazados;
DROP TABLE IF EXISTS workspace.quarantine.tbl_productos_rechazados;
DROP TABLE IF EXISTS workspace.quarantine.tbl_ventas_detalle_rechazados;

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS workspace.silver
COMMENT 'Capa Silver procesados';

CREATE SCHEMA IF NOT EXISTS workspace.quarantine
COMMENT 'Capa quarantena --para revision';


###tbl_representantes

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.silver.tbl_representantes (
  representante STRING,
  ciudad STRING 
  --fotografia DOUBLE
)

""")

In [0]:
df_representantes = spark.table(
    "workspace.bronze.tbl_representantes"
)

# Renombrar columnas
df_representantes = (
    df_representantes
    .withColumnRenamed("Representante", "representante")
    .withColumnRenamed("Ciudad", "ciudad")
    .withColumnRenamed("Fotografía", "fotografia")
)

# Limpieza y estandarización
df_representantes = (
    df_representantes
    .withColumn("representante", F.initcap(F.trim(F.col("representante"))))
    .withColumn("ciudad",F.lower(F.trim(F.col("ciudad"))))
)

# Eliminar columna que no contiene información útil
df_representantes = df_representantes.drop("fotografia")

In [0]:
df_representantes = df_representantes.withColumn("motivo_rechazo",
        F.when(F.col("representante").isNull()| (F.trim(F.col("representante")) == ""),
               F.lit("REPRESENTANTE_NULO"))
       # .when(F.col("ciudad").isNull() | (F.trim(F.col("ciudad")) == ""), # no obligatorio
       #      F.lit("CIUDAD_NULA"))
)
# clasificacion de datos
df_representantes_validos=(df_representantes.filter(F.col("motivo_rechazo").isNull()).drop("motivo_rechazo"))
df_representantes_rechazados=(df_representantes.filter(F.col("motivo_rechazo").isNotNull()))


In [0]:
df_representantes_validos=(df_representantes_validos.dropDuplicates(["representante","ciudad"]))

In [0]:

df_representantes_validos.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tbl_representantes")

In [0]:
#agregamos la el timestamp del rechazo
df_representantes_rechazados = (
    df_representantes_rechazados
    .withColumn("_rejection_timestamp",F.current_timestamp())
)
#cargamos al quarentine
df_representantes_rechazados.write .format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("workspace.quarantine.tbl_representantes_rechazados")

###tbl_productos

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.bronze.tbl_productos (
    `CódigoProducto` STRING,
    `Descripción` STRING,
    `Precio de venta` DECIMAL(12,2),-- contemplara en  las futuras ejecuciones si el datos viene en decimal
    `Costo de venta` DECIMAL(12,2),-- contemplara en las futuras ejecuciones si el datos viene en decimal
    `Almacen` BIGINT,
    `Vendidos` BIGINT
)USING DELTA
TBLPROPERTIES (
    'delta.columnMapping.mode' = 'name'
)

""")

In [0]:
df_productos = spark.table("workspace.bronze.tbl_productos")
# Renombrar columnas
df_productos = (
    df_productos
    .withColumnRenamed("CódigoProducto", "codigo_producto")
    .withColumnRenamed("Descripción", "descripcion")
    .withColumnRenamed("Precio de venta", "precio_venta")
    .withColumnRenamed("Costo de venta", "costo_venta")
    .withColumnRenamed("Almacen", "almacen")
    .withColumnRenamed("Vendidos", "vendidos")
)

# Limpieza y estandarización
df_productos = (
    df_productos.withColumn("codigo_producto",F.upper(F.trim(F.col("codigo_producto"))))
    .withColumn("descripcion",F.initcap(F.trim(F.col("descripcion"))))
    .withColumn("precio_venta",F.col("precio_venta").cast("decimal(12,2)"))
    .withColumn("costo_venta",F.col("costo_venta").cast("decimal(12,2)"))
    .withColumn("almacen",F.col("almacen").cast("bigint"))
)
# Eliminar columna que no contiene información útil
df_productos = df_productos.drop("vendidos")

In [0]:
df_productos = df_productos.withColumn(
    "motivo_rechazo",
    F.when(F.col("codigo_producto").isNull()| (F.trim(F.col("codigo_producto")) == ""),
        F.lit("CODIGO_PRODUCTO_NULO")
    )
    .when(F.col("descripcion").isNull(),
        F.lit("DESCRIPCION_NULA")
    )
    .when( F.col("precio_venta").isNull(),
		F.lit("PRECIO_VENTA_NULO")
    )
    .when(F.col("precio_venta") < 0,
        F.lit("PRECIO_VENTA_NEGATIVO")
    )
    .when(F.col("costo_venta").isNull(),
        F.lit("COSTO_VENTA_NULO")
    )
    .when( F.col("costo_venta") < 0,
        F.lit("COSTO_VENTA_NEGATIVO")
    )
    .when(F.col("almacen") < 0,
        F.lit("ALMACEN_NEGATIVO")
    )
)
# clasificacion de datos
df_productos_validos=(df_productos.filter(F.col("motivo_rechazo").isNull()).drop("motivo_rechazo"))
df_productos_rechazados=(df_productos.filter(F.col("motivo_rechazo").isNotNull()))

In [0]:
df_productos_validos=df_productos_validos.dropDuplicates(["codigo_producto"])

In [0]:

df_productos_validos.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tbl_productos")

In [0]:
#agregamos la el timestamp del rechazo
df_productos_rechazados = (
    df_productos_rechazados
    .withColumn("_rejection_timestamp",F.current_timestamp())
)
#cargamos al quarentine
df_productos_rechazados.write .format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("workspace.quarantine.tbl_productos_rechazados")

###tbl_ventas_detalle

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.silver.tbl_ventas_detalle (
  fecha DATE,
  representante STRING,
  codigo_producto STRING,
  unidades BIGINT
)

""")

In [0]:
df_ventas_detalle=spark.table("workspace.bronze.tbl_ventas_detalle")
# renombrar columnas
df_ventas_detalle = (
    df_ventas_detalle
    .withColumnRenamed("Fecha", "fecha")
    .withColumnRenamed("Representante", "representante")
    .withColumnRenamed("CódigoProducto", "codigo_producto")
    .withColumnRenamed("Unidades", "unidades")
)
# Limpieza y estandarización
df_ventas_detalle = (
    df_ventas_detalle.withColumn("codigo_producto",F.upper(F.trim(F.col("codigo_producto"))))
    .withColumn("representante",F.initcap(F.trim(F.col("representante"))))
    .withColumn("unidades",F.col("unidades").cast("bigint"))
    .withColumn("fecha_validada",F.expr("try_cast(fecha as DATE)"))
    )


In [0]:
df_ventas_detalle = df_ventas_detalle.withColumn(
    "motivo_rechazo",

    F.when( F.col("fecha").isNull(),
        F.lit("FECHA_NULA")
    )
    .when( F.col("fecha_validada").isNull(),
        F.lit("FECHA_INVALIDA")
    )

    .when(F.col("representante").isNull()| (F.trim(F.col("representante")) == ""),
        F.lit("REPRESENTANTE_NULO")
    )
    .when(F.col("codigo_producto").isNull() | (F.trim(F.col("codigo_producto")) == ""),
        F.lit("CODIGO_PRODUCTO_NULO")
    )
    .when(F.col("unidades").isNull(),
        F.lit("UNIDADES_NULAS")
    )
    .when(F.col("unidades") <= 0,
        F.lit("UNIDADES_INVALIDAS")
    )
)
# clasificacion de datos
df_ventas_detalle_validos=(df_ventas_detalle.filter(F.col("motivo_rechazo").isNull()))
df_ventas_detalle_rechazados=(df_ventas_detalle.filter(F.col("motivo_rechazo").isNotNull()))
#reemplazar fecha origina por fecha validada
df_ventas_detalle_validos=df_ventas_detalle_validos.drop("fecha").withColumnRenamed("fecha_validada","fecha")



In [0]:
# ventas sin representante enlazado
df_representantes_silver=spark.table("workspace.silver.tbl_representantes")
ventas_sin_representante=df_ventas_detalle_validos.select("representante").join(df_representantes_silver.select("representante"),
                                                        on="representante",
                                                        how="left_anti")
ventas_sin_representante.show()
total_filas=ventas_sin_representante.count()

# se agrega a la tabla silver representantes
if not ventas_sin_representante.isEmpty() :
    (ventas_sin_representante.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable("workspace.silver.tbl_representantes"))
    print(f"se agregaron {total_filas} representantes")
else:
    print("no hay ventas sin representante")


In [0]:
#ventas sin producto 
ventas_sin_producto=df_ventas_detalle_validos.join(df_productos_validos.select("codigo_producto"),
                                                        on="codigo_producto",
                                                        how="leftanti").withColumn(
                                                            "motivo_rechazo",
                                                            F.lit("PRODUCTO_NO_EXISTE")
    )
                                            

In [0]:

df_productos_silver=spark.table("workspace.silver.tbl_productos")
df_ventas_detalle_validos = (
    df_ventas_detalle_validos
    .join(
        df_productos_silver.select("codigo_producto"),
        on="codigo_producto",
        how="left_semi"
    ).drop("motivo_rechazo")
)

In [0]:
df_ventas_detalle_validos.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("workspace.silver.tbl_ventas_detalle")

In [0]:
#agregamos la el timestamp del rechazo
df_ventas_detalle_rechazados = (
    df_ventas_detalle_rechazados
    .withColumn("_rejection_timestamp",F.current_timestamp())
)
#cargamos al quarentine
df_ventas_detalle_rechazados.write .format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("workspace.quarantine.tbl_ventas_detalle_rechazados")

#agregamos la el timestamp del rechazo
ventas_sin_producto = (
    ventas_sin_producto
    .withColumn("_rejection_timestamp",F.current_timestamp())
)
#añadir casos sin producto

ventas_sin_producto.write.format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable("workspace.quarantine.tbl_ventas_detalle_rechazados")
    

In [0]:
%sql
use workspace.silver;
select *from tbl_representantes;

In [0]:
%sql
use workspace.silver;
select *from tbl_ventas_detalle;